[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Open-Medica/open-medical-skills/blob/main/oms-models/notebooks/04_train_gte_qwen.ipynb)

# 04 - Fine-Tune GTE-Qwen2-1.5B

**OMS Custom Embedding Model Pipeline - Step 4 of 6**

This notebook fine-tunes [GTE-Qwen2-1.5B](https://huggingface.co/Alibaba-NLP/gte-Qwen2-1.5B-instruct) --
a 1.5B parameter instruction-tuned embedding model from Alibaba -- on the validated OMS training pairs.

**Why GTE-Qwen2?**
- 1.5B params -- significantly more capacity than EmbeddingGemma-300M
- Instruction-tuned -- supports task-specific prompts
- Strong MTEB performance, especially on retrieval tasks
- Uses Qwen2 decoder architecture (different inductive biases from Gemma)

**Training approach:**
- **Full fine-tuning** (A100/H100) with MultipleNegativesRankingLoss
- **LoRA fine-tuning** (T4) with QLoRA for memory efficiency
- Gradient checkpointing for reduced VRAM usage
- Instruction-based prompt format

**GPU requirements:** T4 (LoRA only) or A100/H100 (full fine-tuning).

---

## 1. Setup & Dependencies

In [ ]:
!pip install -q sentence-transformers>=3.0.0 torch accelerate datasets pyyaml huggingface_hub
!pip install -q peft bitsandbytes  # For LoRA/QLoRA

In [ ]:
import json
import os
import sys
from pathlib import Path

import torch
from datasets import DatasetDict, load_from_disk, Dataset
from sentence_transformers import SentenceTransformer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. GPU Check & Training Mode Selection

GTE-Qwen2-1.5B requires significantly more VRAM than EmbeddingGemma-300M.
- **A100 (40/80GB)**: Full fine-tuning with batch size 32
- **V100 (16/32GB)**: Full fine-tuning with gradient checkpointing, batch size 8
- **T4 (16GB)**: LoRA/QLoRA only, batch size 4

The notebook auto-detects your GPU and selects the appropriate mode.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU required for GTE-Qwen2-1.5B training.\n"
        "In Colab: Runtime -> Change runtime type -> GPU (T4 for LoRA, A100 for full)"
    )

gpu_name = torch.cuda.get_device_name(0).lower()
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9

if vram_gb >= 40:
    TRAINING_MODE = 'full'
    BATCH_SIZE = 32
    GRADIENT_CHECKPOINTING = False
    USE_LORA = False
    FP16 = True
elif vram_gb >= 24:
    TRAINING_MODE = 'full_gc'
    BATCH_SIZE = 16
    GRADIENT_CHECKPOINTING = True
    USE_LORA = False
    FP16 = True
elif vram_gb >= 14:
    TRAINING_MODE = 'lora'
    BATCH_SIZE = 4
    GRADIENT_CHECKPOINTING = True
    USE_LORA = True
    FP16 = True
else:
    TRAINING_MODE = 'qlora'
    BATCH_SIZE = 2
    GRADIENT_CHECKPOINTING = True
    USE_LORA = True
    FP16 = True

print(f"GPU:                   {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB)")
print(f"Training mode:         {TRAINING_MODE}")
print(f"Batch size:            {BATCH_SIZE}")
print(f"Gradient checkpointing:{GRADIENT_CHECKPOINTING}")
print(f"LoRA:                  {USE_LORA}")
print(f"FP16:                  {FP16}")

if TRAINING_MODE == 'lora':
    print("\nNOTE: Using LoRA fine-tuning. Only ~2% of parameters will be trained.")
    print("For full fine-tuning, use Colab Pro with A100 GPU.")

In [ ]:
# Mount Google Drive for checkpoints (optional)
# from google.colab import drive
# drive.mount('/content/drive')
# OUTPUT_DIR = '/content/drive/MyDrive/oms-models/checkpoints/oms-toolrag-qwen-v1'

OUTPUT_DIR = 'oms-toolrag-qwen-v1'
print(f"Model output directory: {OUTPUT_DIR}")

## 3. Load Training Data

Load the same validated train/val splits from Step 2.

In [ ]:
# Load dataset (same paths as notebook 03)
data_paths = [
    Path('data/processed/oms_training_pairs'),
    Path('../data/processed/oms_training_pairs'),
    Path('/content/drive/MyDrive/oms-models/data/processed/oms_training_pairs'),
]

dataset = None
for data_path in data_paths:
    if data_path.exists():
        dataset = load_from_disk(str(data_path))
        print(f"Loaded dataset from: {data_path}")
        break

if dataset is None:
    for base_dir in [Path('data/processed'), Path('../data/processed')]:
        train_path = base_dir / 'train.jsonl'
        val_path = base_dir / 'val.jsonl'
        if train_path.exists():
            train_records = [json.loads(line) for line in open(train_path) if line.strip()]
            val_records = [json.loads(line) for line in open(val_path) if line.strip()]
            dataset = DatasetDict({
                'train': Dataset.from_list(train_records),
                'validation': Dataset.from_list(val_records),
            })
            print(f"Loaded from JSONL: {base_dir}")
            break

assert dataset is not None, "No training data found. Run notebooks 01 and 02 first."

# Rename for sentence-transformers
train_ds = dataset['train'].rename_columns({'query': 'anchor', 'tool_description': 'positive'})
val_ds = dataset['validation'].rename_columns({'query': 'anchor', 'tool_description': 'positive'})
train_ds = train_ds.select_columns(['anchor', 'positive'])
val_ds = val_ds.select_columns(['anchor', 'positive'])

print(f"Train: {len(train_ds)} pairs")
print(f"Val:   {len(val_ds)} pairs")

## 4. Load Base Model

Load GTE-Qwen2-1.5B. This model uses an instruction-based prompt format for queries.
We configure it to exclude the instruction prompt from document embeddings.

In [ ]:
# HuggingFace login (if needed)
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print("Loaded HF_TOKEN from Colab Secrets")
except (ImportError, Exception):
    pass

# Try local path first
MODEL_NAME = 'Alibaba-NLP/gte-Qwen2-1.5B-instruct'
local_model_paths = [
    'models/gte-qwen2-1.5b',
    '../models/gte-qwen2-1.5b',
    'oms-models/models/gte-qwen2-1.5b',
]

model_path = MODEL_NAME
for local_path in local_model_paths:
    if Path(local_path).exists():
        model_path = local_path
        print(f"Using local model: {local_path}")
        break

print(f"Loading model from: {model_path}")

In [ ]:
model = SentenceTransformer(model_path, trust_remote_code=True)

# Configure instruction-based prompting
# The query gets an instruction prefix; the document does not
model.set_pooling_include_prompt(False)

print(f"Model loaded successfully.")
print(f"\nModel architecture:")
print(model)
print(f"\nEmbedding dimension: {model.get_sentence_embedding_dimension()}")
print(f"Max sequence length: {model.max_seq_length}")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params / 1e6:.1f}M")

## 5. Apply LoRA (if using LoRA mode)

For T4 GPUs with 16GB VRAM, we apply LoRA adapters to reduce the number of trainable parameters
from 1.5B to ~30M (~2%). This makes fine-tuning feasible on consumer GPUs.

In [ ]:
if USE_LORA:
    from peft import LoraConfig, get_peft_model, TaskType

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
        lora_dropout=0.05,
        bias='none',
        task_type=TaskType.FEATURE_EXTRACTION,
    )

    # Apply LoRA to the underlying transformer model
    # sentence-transformers wraps the HF model in model[0].auto_model
    base_model = model[0].auto_model
    peft_model = get_peft_model(base_model, lora_config)
    model[0].auto_model = peft_model

    # Print trainable parameter stats
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"LoRA applied.")
    print(f"  Trainable parameters: {trainable / 1e6:.1f}M ({100*trainable/total:.2f}%)")
    print(f"  Total parameters:     {total / 1e6:.1f}M")
    print(f"  LoRA rank:            {lora_config.r}")
    print(f"  LoRA alpha:           {lora_config.lora_alpha}")
    print(f"  Target modules:       {lora_config.target_modules}")
else:
    print("Full fine-tuning mode -- all parameters are trainable.")
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable / 1e6:.1f}M")

## 6. Enable Gradient Checkpointing (Memory Optimization)

Gradient checkpointing trades compute for memory: recomputes activations during the backward pass
instead of storing them. This reduces VRAM usage by ~40% at the cost of ~20% slower training.

In [ ]:
if GRADIENT_CHECKPOINTING:
    base_model = model[0].auto_model
    if hasattr(base_model, 'gradient_checkpointing_enable'):
        base_model.gradient_checkpointing_enable()
        print("Gradient checkpointing enabled.")
    elif hasattr(base_model, 'base_model') and hasattr(base_model.base_model, 'gradient_checkpointing_enable'):
        # For PEFT-wrapped models
        base_model.base_model.gradient_checkpointing_enable()
        print("Gradient checkpointing enabled (PEFT model).")
    else:
        print("WARNING: Gradient checkpointing not available for this model architecture.")
else:
    print("Gradient checkpointing disabled (sufficient VRAM).")

# VRAM estimate
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    print(f"\nCurrent VRAM usage:")
    print(f"  Allocated: {allocated:.2f} GB")
    print(f"  Reserved:  {reserved:.2f} GB")
    print(f"  Available: {vram_gb - reserved:.2f} GB")

## 7. Configure Training

Set up loss function and training arguments. For GTE-Qwen2, we use standard
MultipleNegativesRankingLoss (not cached) since the batch sizes are already smaller.

In [ ]:
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import (
    BatchSamplers,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.trainer import SentenceTransformerTrainer

# Loss function
loss = MultipleNegativesRankingLoss(model)

# Training arguments
args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=1e-5 if not USE_LORA else 2e-4,  # Higher LR for LoRA
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=FP16,
    gradient_accumulation_steps=4 if BATCH_SIZE <= 8 else 1,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_steps=10,
    report_to='none',
    dataloader_num_workers=2,
)

# Effective batch size
effective_bs = args.per_device_train_batch_size * args.gradient_accumulation_steps
steps_per_epoch = len(train_ds) // effective_bs
total_steps = steps_per_epoch * args.num_train_epochs

print(f"Training configuration:")
print(f"  Training mode:          {TRAINING_MODE}")
print(f"  Epochs:                 {args.num_train_epochs}")
print(f"  Batch size:             {args.per_device_train_batch_size}")
print(f"  Gradient accumulation:  {args.gradient_accumulation_steps}")
print(f"  Effective batch size:   {effective_bs}")
print(f"  Learning rate:          {args.learning_rate}")
print(f"  Warmup ratio:           {args.warmup_ratio}")
print(f"  Estimated steps:        {total_steps}")

## 8. Train

Run training. Expected times:
- **T4 (LoRA)**: ~1-2 hours for 3 epochs
- **A100 (full)**: ~30-45 minutes for 3 epochs

In [ ]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    loss=loss,
)

print(f"Starting {TRAINING_MODE} training...")
print(f"Monitor loss in the output below.\n")

train_result = trainer.train()

print(f"\nTraining complete!")
print(f"  Final train loss: {train_result.training_loss:.4f}")
print(f"  Total steps: {train_result.global_step}")
print(f"  Training time: {train_result.metrics.get('train_runtime', 0):.0f}s")

if torch.cuda.is_available():
    peak_mem = torch.cuda.max_memory_allocated(0) / 1e9
    print(f"  Peak VRAM: {peak_mem:.2f} GB")

In [ ]:
# Plot training loss curve
try:
    import matplotlib.pyplot as plt

    logs = trainer.state.log_history
    train_losses = [(log['step'], log['loss']) for log in logs if 'loss' in log]
    eval_losses = [(log['step'], log['eval_loss']) for log in logs if 'eval_loss' in log]

    fig, ax = plt.subplots(1, 1, figsize=(10, 5))

    if train_losses:
        steps, losses = zip(*train_losses)
        ax.plot(steps, losses, label='Train Loss', alpha=0.7)

    if eval_losses:
        steps, losses = zip(*eval_losses)
        ax.plot(steps, losses, label='Eval Loss', marker='o', markersize=4)

    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.set_title(f'GTE-Qwen2-1.5B Fine-Tuning ({TRAINING_MODE})')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Could not plot loss curve: {e}")

## 9. Quick Evaluation

Same sanity check as notebook 03: encode test queries and corpus, show top-5 retrieval results.

In [ ]:
import random
from sentence_transformers.util import cos_sim

# Load test data
test_ds_raw = None
if 'test' in dataset:
    test_ds_raw = dataset['test']
else:
    for path in [Path('data/processed/test.jsonl'), Path('../data/processed/test.jsonl')]:
        if path.exists():
            test_records = [json.loads(line) for line in open(path) if line.strip()]
            test_ds_raw = Dataset.from_list(test_records)
            break

if test_ds_raw is None:
    test_ds_raw = dataset['validation']

# Sample queries and build corpus
random.seed(42)
test_indices = random.sample(range(len(test_ds_raw)), min(10, len(test_ds_raw)))
test_queries = [test_ds_raw[i]['query'] for i in test_indices]
test_ground_truth = [test_ds_raw[i]['tool_name'] for i in test_indices]

all_records_for_corpus = list(dataset['train']) + list(dataset['validation'])
if 'test' in dataset:
    all_records_for_corpus += list(dataset['test'])

corpus = {}
for r in all_records_for_corpus:
    if r['tool_name'] not in corpus:
        corpus[r['tool_name']] = r['tool_description']

corpus_names = list(corpus.keys())
corpus_descs = list(corpus.values())

# Encode with instruction prompt for queries
query_prompt = "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: "
query_embeddings = model.encode(
    test_queries,
    convert_to_tensor=True,
    show_progress_bar=True,
    prompt=query_prompt,
)
corpus_embeddings = model.encode(
    corpus_descs,
    convert_to_tensor=True,
    show_progress_bar=True,
    batch_size=16,
)

similarities = cos_sim(query_embeddings, corpus_embeddings)

print(f"\n{'='*80}")
print(f"RETRIEVAL RESULTS (GTE-Qwen2 {TRAINING_MODE})")
print(f"{'='*80}")

hits = 0
for i, (query, gt_tool) in enumerate(zip(test_queries, test_ground_truth)):
    scores, indices = similarities[i].topk(5)
    top5_names = [corpus_names[idx] for idx in indices]
    is_hit = gt_tool in top5_names
    if is_hit:
        hits += 1

    print(f"\n[{'HIT' if is_hit else 'MISS'}] Query: {query[:80]}")
    print(f"  Ground truth: {gt_tool}")
    for rank, (name, score) in enumerate(zip(top5_names, scores), 1):
        marker = ' <<' if name == gt_tool else ''
        print(f"    {rank}. {name} (sim={score:.4f}){marker}")

print(f"\nRecall@5: {hits}/{len(test_queries)} ({100*hits/len(test_queries):.1f}%)")

## 10. Save Model

Save the fine-tuned model. For LoRA models, save both the adapter weights and the merged model.

In [ ]:
# Save the full model (or merged LoRA model)
if USE_LORA:
    # Save LoRA adapter weights separately
    adapter_dir = f"{OUTPUT_DIR}-lora-adapter"
    peft_model = model[0].auto_model
    peft_model.save_pretrained(adapter_dir)
    print(f"LoRA adapter saved to: {adapter_dir}")

    # Merge LoRA weights into base model and save
    print("Merging LoRA weights into base model...")
    merged_model = peft_model.merge_and_unload()
    model[0].auto_model = merged_model

model.save(OUTPUT_DIR)
print(f"Full model saved to: {OUTPUT_DIR}")

# List saved files
saved_files = list(Path(OUTPUT_DIR).glob('*'))
total_size = sum(f.stat().st_size for f in saved_files if f.is_file()) / 1e9
print(f"\nTotal size: {total_size:.2f} GB")
for f in sorted(saved_files):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        print(f"  {f.name} ({size_mb:.1f} MB)")

In [ ]:
# Optional: Push to HuggingFace Hub

# HF_REPO = 'intelmedica/oms-toolrag-qwen-v1'
# model.push_to_hub(
#     HF_REPO,
#     private=True,
#     commit_message='Fine-tuned GTE-Qwen2-1.5B on OMS medical tool retrieval data',
# )
# print(f"Model pushed to: https://huggingface.co/{HF_REPO}")

---

## Next Steps

Proceed to **05_evaluate_models.ipynb** to benchmark both fine-tuned models against baselines.

**Files produced by this notebook:**
- `oms-toolrag-qwen-v1/` -- Fine-tuned model weights (full or merged LoRA)
- `oms-toolrag-qwen-v1-lora-adapter/` -- LoRA adapter weights (if using LoRA mode)
- Checkpoint directories under `oms-toolrag-qwen-v1/checkpoint-*/`